In [4]:
from abc import ABC, abstractmethod
from typing import Optional, List, Type, Tuple, Dict
import math

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.axes._axes import Axes
import torch
import torch.nn as nn
import torch.distributions as D
from torch.func import vmap, jacrev
from tqdm import tqdm

import yaml 

from pathlib import Path
from torch.utils.data import DataLoader

import os
import hydra
from omegaconf import OmegaConf

# import seaborn as sns
# from sklearn.datasets import make_moons, make_circles
# from torchvision import datasets, transforms
# from torchvision.utils import make_grid

In [5]:
from boltz.data.types import MSA, Connection, Input, Record, Structure
import json

In [6]:
# structure

record_id = "7znt"
target_dir = Path("/Users/nvithayapale/Desktop/boltz-finetune/data/processed_ternary_data/processed_structures")
structure = np.load(target_dir / "structures" / f"{record_id}.npz")
record = (target_dir / "records" / f"{record_id}.npz")

In [7]:
with open(target_dir / "records" / f"{record_id}.json", "r") as f:
    json_loaded = json.load(f)
    record_parsed = Record.from_dict(json_loaded)

In [8]:
structure_parsed = Structure(
    atoms=structure["atoms"],
    bonds=structure["bonds"],
    residues=structure["residues"],
    chains=structure["chains"],
    connections=structure["connections"].astype(Connection),
    interfaces=structure["interfaces"],
    mask=structure["mask"]
)

msa_dir = Path("/Users/nvithayapale/Desktop/boltz-finetune/data/processed_ternary_data/processed_msa")
loaded_msa = np.load(msa_dir / f"{record_id}.npz")
msas = {}
for chain in record_parsed.chains:
    msa_id = chain.msa_id
    # Load the MSA for this chain, if any
    if msa_id != -1 and msa_id != "":
        msa = np.load(msa_dir / f"{msa_id}.npz")
        msas[chain.chain_id] = MSA(**msa)

input_parsed = Input(structure_parsed, msas, record_parsed)
# input_parsed

In [9]:
print("structure", [key for key in input_parsed.structure.__dict__.keys()])
print("msa", [key for key in input_parsed.msa.keys()])

structure ['atoms', 'bonds', 'residues', 'chains', 'connections', 'interfaces', 'mask']
msa [0, 1, 2, 3, 4]


## Feature Processing Demonstration

In [10]:
# from boltz.data.module.training import Dataset
from boltz.data.crop.boltz import BoltzCropper
from torch import Tensor
from boltz.data.types import Input
from boltz.data.tokenize.boltz import BoltzTokenizer
from boltz.data.feature.featurizer import BoltzFeaturizer
from boltz.data.module.training import collate

In [11]:
tokenizer = BoltzTokenizer()
tokenized_data = tokenizer.tokenize(input_parsed)
tokenized_data.__dict__.keys()

dict_keys(['tokens', 'bonds', 'structure', 'msa'])

In [12]:
# REFERENCE:
# res_num = len(chain.residues)
# atom_num = sum(len(res.atoms) for res in chain.residues): all atoms in that residue

tokenized_data.tokens[3]['disto_idx']

27

In [13]:
coord_data = [input_parsed.structure.atoms[0:8]['coords'], input_parsed.structure.atoms[8:16]['coords']]
dummy_coord = np.concatenate(coord_data, axis=1).reshape(-1,3)
(dummy_coord[:, None, :] - dummy_coord[:, :, None]).shape

(16, 3, 3)

## Pre-loading into DataLoader

In [14]:
# Move preprocessing out of main process

class SimpleTrainingDataset(torch.utils.data.Dataset):
    """Base iterable dataset."""

    def __init__(
        self,
        input_data: Input | List[Input],
        samples_per_epoch: int,
        symmetries: dict,
        max_atoms: int,
        max_tokens: int,
        max_seqs: int,
        pad_to_max_atoms: bool = False,
        pad_to_max_tokens: bool = False,
        pad_to_max_seqs: bool = False,
        atoms_per_window_queries: int = 32,
        min_dist: float = 2.0,
        max_dist: float = 22.0,
        num_bins: int = 64,
        binder_pocket_conditioned_prop: Optional[float] = 0.0,
        binder_pocket_cutoff: Optional[float] = 6.0,
        binder_pocket_sampling_geometric_p: Optional[float] = 0.0,
        return_symmetries: Optional[bool] = False,
    ) -> None:
        """Initialize the training dataset."""
        super().__init__()
        self.input_data = input_data
        self.symmetries = symmetries
        self.max_tokens = max_tokens
        self.max_seqs = max_seqs
        self.max_atoms = max_atoms
        self.pad_to_max_tokens = pad_to_max_tokens
        self.pad_to_max_atoms = pad_to_max_atoms
        self.pad_to_max_seqs = pad_to_max_seqs
        self.atoms_per_window_queries = atoms_per_window_queries
        self.min_dist = min_dist
        self.max_dist = max_dist
        self.num_bins = num_bins
        self.binder_pocket_conditioned_prop = binder_pocket_conditioned_prop
        self.binder_pocket_cutoff = binder_pocket_cutoff
        self.binder_pocket_sampling_geometric_p = binder_pocket_sampling_geometric_p
        self.return_symmetries = return_symmetries
        self.tokenizer = BoltzTokenizer()
        self.featurizer = BoltzFeaturizer()
        self.cropper = BoltzCropper()
        self.samples_per_epoch = samples_per_epoch

    def __getitem__(self, idx: int) -> dict[str, Tensor]:
        """Get an item from the dataset.

        Parameters
        ----------
        idx : int
            The data index.

        Returns
        -------
        dict[str, Tensor]
            The sampled data features.
        """

        # Don't pre-loading everything into datasets first
        # Use index in DataSet and load via DataLoader

        try:
            tokenized = self.tokenizer.tokenize(self.input_data)
        except Exception as e:
            print(f"Tokenizer failed on {self.input_data.record.id} with error {e}. Skipping.")
            return self.__getitem__(idx)

        # Compute crop
        try:
            if self.max_tokens is not None:
                tokenized = self.cropper.crop(
                    tokenized,
                    max_atoms=self.max_atoms,
                    max_tokens=self.max_tokens,
                    random=np.random,
                    chain_id=None,
                    interface_id=None,
                )
        except Exception as e:
            print(f"Cropper failed on {self.input_data.record.id} with error {e}. Skipping.")
            return self.__getitem__(idx)

        # Check if there are tokens
        if len(tokenized.tokens) == 0:
            msg = "No tokens in cropped structure."
            raise ValueError(msg)

        # Compute features
        try:
            features = self.featurizer.process(
                tokenized,
                training=True,
                max_atoms=self.max_atoms if self.pad_to_max_atoms else None,
                max_tokens=self.max_tokens if self.pad_to_max_tokens else None,
                max_seqs=self.max_seqs,
                pad_to_max_seqs=self.pad_to_max_seqs,
                symmetries=self.symmetries,
                atoms_per_window_queries=self.atoms_per_window_queries,
                min_dist=self.min_dist,
                max_dist=self.max_dist,
                num_bins=self.num_bins,
                compute_symmetries=self.return_symmetries,
                binder_pocket_conditioned_prop=self.binder_pocket_conditioned_prop,
                binder_pocket_cutoff=self.binder_pocket_cutoff,
                binder_pocket_sampling_geometric_p=self.binder_pocket_sampling_geometric_p,
            )
        except Exception as e:
            print(f"Featurizer failed on {self.input_data.record.id} with error {e}. Skipping.")
            return self.__getitem__(idx)

        return features

    def __len__(self) -> int:
        """Get the length of the dataset.

        Returns
        -------
        int
            The length of the dataset.

        """
        return self.samples_per_epoch

In [15]:
# Load configuration file

def load_config(config_path):
    with open(config_path, 'r') as f:
        return yaml.safe_load(f)

# Path to your config file
config_path = Path("/Users/nvithayapale/Desktop/boltz-finetune/scripts/train/configs/full.yaml")
config = load_config(config_path)

# Extract dataset parameters from config

symmetries = config.get('symmetries', "/Users/nvithayapale/Desktop/boltz-finetune/scripts/train/symmetry.pkl")
max_tokens = config.get('max_tokens', 512)
max_atoms = config.get('max_atoms', 4608)
max_seqs = config.get('max_seqs', 2048)
pad_to_max_atoms = config.get('pad_to_max_atoms', True)
pad_to_max_tokens = config.get('pad_to_max_tokens', True)
pad_to_max_seqs = config.get('pad_to_max_seqs', True)
atoms_per_window_queries = config.get('atoms_per_window_queries', 32)
min_dist = float(config.get('min_dist', 2.0))
max_dist = float(config.get('max_dist', 22.0))
num_bins = int(config.get('num_bins', 64))
binder_pocket_conditioned_prop = config.get('train_binder_pocket_conditioned_prop', 0.3)
binder_pocket_cutoff = config.get('binder_pocket_cutoff', 6.0)
binder_pocket_sampling_geometric_p = config.get('binder_pocket_sampling_geometric_p', 0.3)
return_symmetries = config.get('return_train_symmetries', True)
samples_per_epoch = 1

In [16]:
simple_dataset = SimpleTrainingDataset(
    input_parsed, 
    samples_per_epoch,
    symmetries, 
    max_atoms, 
    max_tokens, 
    max_seqs, 
    pad_to_max_atoms, 
    pad_to_max_tokens, 
    pad_to_max_seqs, 
    atoms_per_window_queries, 
    min_dist, 
    max_dist,
    num_bins, 
    binder_pocket_conditioned_prop, 
    binder_pocket_cutoff, 
    binder_pocket_sampling_geometric_p, 
    return_symmetries
)

In [17]:
chains = [c for c in input_parsed.record.chains if c.valid]
interfaces = [i for i in input_parsed.record.interfaces if i.valid]

In [18]:
data_config = {
    "batch_size": 1,
    "num_workers": 0,
    "pin_memory": False,
    "shuffle": False,
    "collate_fn": collate,
}

In [19]:
simple_dataloader = DataLoader(
    simple_dataset,
    batch_size=data_config["batch_size"],
    num_workers=data_config["num_workers"],
    pin_memory=data_config["pin_memory"],
    shuffle=False,
    collate_fn=collate,
)

In [20]:
# max_tokens, max_atom

In [21]:
# for batch in simple_dataloader:
#     for key, value in batch.items():
#         print(key)
#         if isinstance(value, torch.Tensor):
#             print(value.shape)
#         else:
#             print("Non Tensor")

## Architecture

In [22]:
from boltz.model.modules.utils import LinearNoBias
import boltz.model.layers.initialize as init
from boltz.model.modules.transformers import DiffusionTransformer
from boltz.model.modules.encoders import get_indexing_matrix, single_to_keys
from functools import partial
from torch.nn.functional import one_hot

In [23]:
feats = next(iter(simple_dataloader))

In [24]:
print("Check Input Shape")
print(feats["ref_pos"].shape) 
print(feats["ref_charge"].shape)
print(feats["atom_pad_mask"].shape)
print(feats["ref_element"].shape)
print(feats["ref_atom_name_chars"].shape)

B, N, _ = feats["ref_pos"].shape

atom_feats = torch.cat(
    [
        feats["ref_pos"], # B, N
        feats["ref_charge"].unsqueeze(-1), # B, N, 1
        feats["atom_pad_mask"].unsqueeze(-1), # B, N, 1
        feats["ref_element"], 
        feats["ref_atom_name_chars"].reshape(B, N, 4 * 64),
    ],
    dim=-1,
)

print(atom_feats.shape)

Check Input Shape
torch.Size([1, 4608, 3])
torch.Size([1, 4608])
torch.Size([1, 4608])
torch.Size([1, 4608, 128])
torch.Size([1, 4608, 4, 64])
torch.Size([1, 4608, 389])


In [50]:
# Load config directly from file (when not using @hydra.main decorator)
config_path = os.path.expanduser("~/Desktop/boltz-finetune/scripts/train/configs/full_finetune.yaml")
cfg = OmegaConf.load(config_path)

# Print config details
print(f"Model type: {cfg.model._target_}")
print(f"Token dimension: {cfg.model.token_s}")

# Input embeddings
full_embedder_args = {
    "atom_s": cfg.model.atom_s,
    "atom_z": cfg.model.atom_z,
    "token_s": cfg.model.token_s,
    "token_z": cfg.model.token_z,
    "atoms_per_window_queries": cfg.model.atoms_per_window_queries,
    "atoms_per_window_keys": cfg.model.atoms_per_window_keys,
    "atom_feature_dim": cfg.model.atom_feature_dim,
    "no_atom_encoder": 3,
    **cfg.model.embedder_args,
}
print(full_embedder_args)

Model type: boltz.model.model.Boltz1
Token dimension: 384
{'atom_s': 128, 'atom_z': 16, 'token_s': 384, 'token_z': 128, 'atoms_per_window_queries': 32, 'atoms_per_window_keys': 128, 'atom_feature_dim': 389, 'no_atom_encoder': 3, 'atom_encoder_depth': 3, 'atom_encoder_heads': 4}


In [76]:
from einops.layers.torch import Rearrange

class AttentionPairBias(nn.Module):

    def __init__(self, c_s, c_z, num_heads, inf=1e6, initial_norm=True):
        super().__init__()
        assert c_s % num_heads == 0

        self.c_s = c_s
        self.c_z = c_z
        self.num_heads = num_heads
        self.inf = inf
        self.initial_norm = initial_norm
        self.head_dim = c_s // num_heads
        if self.initial_norm:
            self.norm_s = nn.LayerNorm(c_s)

        self.proj_q = nn.Linear(c_s, c_s)
        self.proj_k = nn.Linear(c_s, c_s, bias=False)
        self.proj_v = nn.Linear(c_s, c_s, bias=False)
        self.proj_g = nn.Linear(c_s, c_s, bias=False)

        self.proj_z = nn.Sequential(
            nn.LayerNorm(c_z),
            nn.Linear(c_z, num_heads, bias=False),
            Rearrange("b ... h -> b h ..."),
        )

        self.proj_o = nn.Linear(c_s, c_s, bias=False)
        init.final_init_(self.proj_o.weight)
    
    def forward(
        self,
        s: Tensor,
        z: Tensor,
        mask: Tensor,
        multiplicity: int = 1,
        to_keys=None,
        model_cache=None,
    ) -> Tensor:
        
        B = s.shape[0]

        # Layer norms
        if self.initial_norm:
            s = self.norm_s(s)

        if to_keys is not None:
            k_in = to_keys(s)
            mask = to_keys(mask.unsqueeze(-1)).squeeze(-1)
        else:
            k_in = s
        
        # Compute projections
        q = self.proj_q(s).view(B, -1, self.num_heads, self.head_dim)
        k = self.proj_k(k_in).view(B, -1, self.num_heads, self.head_dim)
        v = self.proj_v(k_in).view(B, -1, self.num_heads, self.head_dim) # (B, N)

        # Caching z projection during diffusion roll-out
        if model_cache is None or "z" not in model_cache:
            z = self.proj_z(z)

            if model_cache is not None:
                model_cache["z"] = z
        else:
            z = model_cache["z"]

        # (B * multiplicity, N, N, D)
        z = z.repeat_interleave(multiplicity, 0)

        g = self.proj_g(s).sigmoid()

        # Autocast on transformer
        with torch.autocast("cuda", enabled=False):
            # Compute attention weights
            # similar to the scaled dot product attention

            attn = torch.einsum("bihd,bjhd->bhij", q.float(), k.float())
            # (B, num_head, N, N)
            attn = attn / (self.head_dim**0.5) + z.float() # (B, H,  N, N)
            attn = attn + (1 - mask[:, None, None].float()) * -self.inf
            attn = attn.softmax(dim=-1)

            # Compute output
            o = torch.einsum("bhij,bjhd->bihd", attn, v.float()).to(v.dtype)
        
        # (B, N, D)
        o = o.reshape(B, -1, self.c_s)
        o = self.proj_o(g * o)

        return o

In [77]:
from boltz.model.modules.transformers import ConditionedTransitionBlock
from boltz.model.modules.transformers import AdaLN

In [79]:
dim=cfg.model.atom_s
dim_single_cond=cfg.model.atom_s
dim_pairwise=cfg.model.atom_z

B = 1
max_atom = 4608
max_token = 512

a = torch.randn(B, max_atom, dim)
c = torch.randn(B, max_atom, dim)
p = torch.randn(B, max_atom, max_atom, dim_pairwise)

attention_pair_bias = AttentionPairBias(
    c_s=dim, 
    c_z=dim_pairwise, 
    num_heads=16
)
b = AdaLN(dim, dim_single_cond)(a, c)
atom_mask = feats["atom_pad_mask"].bool()

result = attention_pair_bias(
    s = b,
    z = p,
    mask = atom_mask
)
print(result.shape)

torch.Size([1, 4608, 128])


In [ ]:
class DiffusionTransformerLayer(nn.Module):
    """Diffusion Transformer Layer"""

    def __init__(
        self,
        heads,
        dim=384,
        dim_single_cond=None,
        dim_pairwise=128,
    ):
        """Initialize the diffusion transformer layer.

        Parameters
        ----------
        heads : int
            The number of heads.
        dim : int, optional
            The dimension, by default 384
        dim_single_cond : int, optional
            The single condition dimension, by default None
        dim_pairwise : int, optional
            The pairwise dimension, by default 128

        """
        super().__init__()

        dim_single_cond = default(dim_single_cond, dim)

        self.adaln = AdaLN(dim, dim_single_cond)

        # Attention Pair Bias
        self.pair_bias_attn = AttentionPairBias(
            c_s=dim, c_z=dim_pairwise, num_heads=heads, initial_norm=False
        )

        self.output_projection_linear = Linear(dim_single_cond, dim)
        nn.init.zeros_(self.output_projection_linear.weight)
        nn.init.constant_(self.output_projection_linear.bias, -2.0)

        self.output_projection = nn.Sequential(
            self.output_projection_linear, nn.Sigmoid()
        )
        self.transition = ConditionedTransitionBlock(
            dim_single=dim, dim_single_cond=dim_single_cond
        )

    # atom-level
    def forward(
        self,
        a,
        s,
        z,
        mask=None,
        to_keys=None,
        multiplicity=1,
        layer_cache=None,
    ):
        # a = q, s = c, z = p
        b = self.adaln(a, s)
        b = self.adaln(a, s)
        b = self.pair_bias_attn(
            s=b,
            z=z,
            mask=mask,
            multiplicity=multiplicity,
            to_keys=to_keys,
            model_cache=layer_cache,
        )
        b = self.output_projection(s) * b

        # NOTE: Added residual connection!
        a = a + b
        a = a + self.transition(a, s)
        return a

In [26]:
from boltz.model.modules.transformers import DiffusionTransformer

class AtomTransformer(nn.Module):
    """Atom Transformer"""

    def __init__(
        self,
        attn_window_queries=None,
        attn_window_keys=None,
        **diffusion_transformer_kwargs,
    ):
        """Initialize the atom transformer.

        Parameters
        ----------
        attn_window_queries : int, optional
            The attention window queries, by default None
        attn_window_keys : int, optional
            The attention window keys, by default None
        diffusion_transformer_kwargs : dict
            The diffusion transformer keyword arguments

        """
        super().__init__()
        self.attn_window_queries = attn_window_queries
        self.attn_window_keys = attn_window_keys
        self.diffusion_transformer = DiffusionTransformer(
            **diffusion_transformer_kwargs
        )

    def forward(
        self,
        q,
        c,
        p,
        to_keys=None,
        mask=None,
        multiplicity=1,
        model_cache=None,
    ):
        W = self.attn_window_queries
        H = self.attn_window_keys

        if W is not None:
            B, N, D = q.shape
            NW = N // W

            # reshape tokens: B * (num_res_per_window)
            q = q.view((B * NW, W, -1))
            c = c.view((B * NW, W, -1))
            if mask is not None:
                # mask is B * (N // W), W
                mask = mask.view(B * NW, W)
            p = p.view((p.shape[0] * NW, W, H, -1))

            # get to keys lambda x: to_keys(x.view(B, NW * W, -1)).view(B * NW, H, -1)
            to_keys_new = lambda x: to_keys(x.view(B, NW * W, -1)).view(B * NW, H, -1)
        else:
            to_keys_new = None

        # main transformer
        q = self.diffusion_transformer(
            a=q,
            s=c,
            z=p,
            mask=mask.float(),
            multiplicity=multiplicity,
            to_keys=to_keys_new,
            model_cache=model_cache,
        )

        if W is not None:
            q = q.view((B, NW * W, D))

        return q


In [65]:
class AtomAttentionEncoder(nn.Module):
    """Atom attention encoder."""

    def __init__(
        self,
        atom_s,
        atom_z,
        token_s,
        token_z,
        atoms_per_window_queries,
        atoms_per_window_keys,
        atom_feature_dim,
        atom_encoder_depth=3,
        atom_encoder_heads=4,
        structure_prediction=True,
        activation_checkpointing=False,
    ):
        """Initialize the atom attention encoder.

        Parameters
        ----------
        atom_s : int
            The atom single representation dimension.
        atom_z : int
            The atom pair representation dimension.
        token_s : int
            The single representation dimension.
        token_z : int
            The pair representation dimension.
        atoms_per_window_queries : int
            The number of atoms per window for queries.
        atoms_per_window_keys : int
            The number of atoms per window for keys.
        atom_feature_dim : int
            The atom feature dimension.
        atom_encoder_depth : int, optional
            The number of transformer layers, by default 3.
        atom_encoder_heads : int, optional
            The number of transformer heads, by default 4.
        structure_prediction : bool, optional
            Whether it is used in the diffusion module, by default True.
        activation_checkpointing : bool, optional
            Whether to use activation checkpointing, by default False.

        """
        super().__init__()

        self.embed_atom_features = LinearNoBias(atom_feature_dim, atom_s)
        self.embed_atompair_ref_pos = LinearNoBias(3, atom_z)
        self.embed_atompair_ref_dist = LinearNoBias(1, atom_z)
        self.embed_atompair_mask = LinearNoBias(1, atom_z)
        self.atoms_per_window_queries = atoms_per_window_queries
        self.atoms_per_window_keys = atoms_per_window_keys

        self.structure_prediction = structure_prediction
        if structure_prediction:
            # from token_s, atom_s
            self.s_to_c_trans = nn.Sequential(
                nn.LayerNorm(token_s), LinearNoBias(token_s, atom_s)
            )
            init.final_init_(self.s_to_c_trans[1].weight)

            self.z_to_p_trans = nn.Sequential(
                nn.LayerNorm(token_z), LinearNoBias(token_z, atom_z)
            )
            init.final_init_(self.z_to_p_trans[1].weight)
            
            # r to q
            self.r_to_q_trans = LinearNoBias(10, atom_s)
            init.final_init_(self.r_to_q_trans.weight)

        self.c_to_p_trans_k = nn.Sequential(
            nn.ReLU(),
            LinearNoBias(atom_s, atom_z),
        )
        init.final_init_(self.c_to_p_trans_k[1].weight)

        self.c_to_p_trans_q = nn.Sequential(
            nn.ReLU(),
            LinearNoBias(atom_s, atom_z),
        )
        init.final_init_(self.c_to_p_trans_q[1].weight)

        self.p_mlp = nn.Sequential(
            nn.ReLU(),
            LinearNoBias(atom_z, atom_z),
            nn.ReLU(),
            LinearNoBias(atom_z, atom_z),
            nn.ReLU(),
            LinearNoBias(atom_z, atom_z),
        )
        init.final_init_(self.p_mlp[5].weight)
        
        self.atom_encoder = AtomTransformer(
            dim=atom_s,
            dim_single_cond=atom_s,
            dim_pairwise=atom_z,
            attn_window_queries=atoms_per_window_queries,
            attn_window_keys=atoms_per_window_keys,
            depth=atom_encoder_depth,
            heads=atom_encoder_heads,
            activation_checkpointing=activation_checkpointing,
        )

        self.atom_to_token_trans = nn.Sequential(
            LinearNoBias(atom_s, 2 * token_s if structure_prediction else token_s),
            nn.ReLU(),
        )

    def forward(
        self,
        feats,
        s_trunk=None,
        z=None,
        r=None,
        multiplicity=1,
        model_cache=None,
    ):
        # N = number_of_atoms
        B, N, _ = feats["ref_pos"].shape
        atom_mask = feats["atom_pad_mask"].bool()

        layer_cache = None
        if model_cache is not None:
            cache_prefix = "atomencoder"
            if cache_prefix not in model_cache:
                model_cache[cache_prefix] = {}
            layer_cache = model_cache[cache_prefix]

        if model_cache is None or len(layer_cache) == 0:
            # either model is not using the cache or it is the first time running it
            atom_ref_pos = feats["ref_pos"]
            atom_uid = feats["ref_space_uid"]
            atom_feats = torch.cat(
                [
                    atom_ref_pos, # B, N
                    feats["ref_charge"].unsqueeze(-1), # B, N, 1
                    feats["atom_pad_mask"].unsqueeze(-1), # B, N, 1
                    feats["ref_element"], 
                    feats["ref_atom_name_chars"].reshape(B, N, 4 * 64),
                ],
                dim=-1,
            )

            c = self.embed_atom_features(atom_feats)

            # NOTE: we are already creating the windows to make it more efficient

            # atoms_per_window_keys
            W, H = self.atoms_per_window_queries, self.atoms_per_window_keys
            B, N = c.shape[:2]
            K = N // W

            # sequence-local attention by window
            keys_indexing_matrix = get_indexing_matrix(N // W, W, H, c.device)
            to_keys = partial(
                single_to_keys, indexing_matrix=keys_indexing_matrix, W=W, H=H
            ) # Shape: [2K, h*K]

            print("atom_ref_pos", atom_ref_pos.shape)
            atom_ref_pos_queries = atom_ref_pos.view(B, K, W, 1, 3) # B, N, 3 -> B, N // W, W, 1, 3
            atom_ref_pos_keys = to_keys(atom_ref_pos).view(B, K, 1, H, 3) # B, N, 3 -> B, N // W, 1, H, 3

            d = atom_ref_pos_keys - atom_ref_pos_queries 
            d_norm = torch.sum(d * d, dim=-1, keepdim=True)
            d_norm = 1 / (1 + d_norm)
            
            atom_mask_queries = atom_mask.view(B, K, W, 1)
            atom_mask_keys = (
                to_keys(atom_mask.unsqueeze(-1).float()).view(B, K, 1, H).bool()
            )
            atom_uid_queries = atom_uid.view(B, K, W, 1)
            atom_uid_keys = (
                to_keys(atom_uid.unsqueeze(-1).float()).view(B, K, 1, H).long()
            )

            # Create validity mask (where atoms belong to same region and are not padding)
            v = (
                (
                    atom_mask_queries
                    & atom_mask_keys
                    & (atom_uid_queries == atom_uid_keys)
                )
                .float()
                .unsqueeze(-1)
            )
            # [B, K, W, H, 1]

            print("d before", d.shape)
            p = self.embed_atompair_ref_pos(d) * v
            p = p + self.embed_atompair_ref_dist(d_norm) * v
            p = p + self.embed_atompair_mask(v) * v

            print("p before", p.shape)
            q = c
            
            if self.structure_prediction:
                # run only in structure model not in initial encoding
                atom_to_token = feats["atom_to_token"].float()

                s_to_c = self.s_to_c_trans(s_trunk)
                s_to_c = torch.bmm(atom_to_token, s_to_c)
                c = c + s_to_c

                atom_to_token_queries = atom_to_token.view(
                    B, K, W, atom_to_token.shape[-1]
                )
                atom_to_token_keys = to_keys(atom_to_token)
                z_to_p = self.z_to_p_trans(z)
                z_to_p = torch.einsum(
                    "bijd,bwki,bwlj->bwkld",
                    z_to_p,
                    atom_to_token_queries,
                    atom_to_token_keys,
                )
                p = p + z_to_p

            print("c before", c.shape)
            print("c to p trans q", c.view(B, K, W, 1, c.shape[-1]).shape)
            print("p shape", p.shape)
            p = p + self.c_to_p_trans_q(c.view(B, K, W, 1, c.shape[-1]))
            p = p + self.c_to_p_trans_k(to_keys(c).view(B, K, 1, H, c.shape[-1]))
            p = p + self.p_mlp(p)

            if model_cache is not None:
                layer_cache["q"] = q
                layer_cache["c"] = c
                layer_cache["p"] = p
                layer_cache["to_keys"] = to_keys

        else:
            q = layer_cache["q"]
            c = layer_cache["c"]
            p = layer_cache["p"]
            to_keys = layer_cache["to_keys"]

        if self.structure_prediction:
            # only here the multiplicity kicks in because we use the different positions r
            q = q.repeat_interleave(multiplicity, 0)
            r_input = torch.cat(
                [r, torch.zeros((B * multiplicity, N, 7)).to(r)],
                dim=-1,
            )
            r_to_q = self.r_to_q_trans(r_input)
            q = q + r_to_q

        c = c.repeat_interleave(multiplicity, 0)
        atom_mask = atom_mask.repeat_interleave(multiplicity, 0)

        print("q shape", q.shape)
        print("atom mask", atom_mask.shape)
        print("p shape", p.shape)
        print("H", atoms_per_window_queries, "W", self.atoms_per_window_keys, "N", N)

        # (q,c,p)
        # Atom Transformer
        q = self.atom_encoder(
            q=q, # q = B, N_atom, atom_s
            mask=atom_mask,
            c=c,  # c = B, N_atom, atom_s
            p=p,  # p = B, N_atom, N_atom, atom_z
            multiplicity=multiplicity,
            to_keys=to_keys,
            model_cache=layer_cache,
        )
        
        q_to_a = self.atom_to_token_trans(q)

        # Aggregate atom-level to token-level
        atom_to_token = feats["atom_to_token"].float()
        atom_to_token = atom_to_token.repeat_interleave(multiplicity, 0)
        atom_to_token_mean = atom_to_token / (
            atom_to_token.sum(dim=1, keepdim=True) + 1e-6
        )
        print("q_to_a", q_to_a.shape) # (1, 4608, 384)
        # 1, 4608, 512
        a = torch.bmm(atom_to_token_mean.transpose(1, 2), q_to_a)
        return a, q, c, p, to_keys

In [66]:
N = feats['ref_pos'].size(1)
H = full_embedder_args["atoms_per_window_keys"]
W = full_embedder_args["atoms_per_window_queries"]
K = (N // W) # number of keys (N_atoms / N_atoms_per_window_per_key)

h = H // (W // 2)
# assert h % 2 == 0

arange = torch.arange(2 * K)
index = ((arange.unsqueeze(0) - arange.unsqueeze(1)) + h // 2).clamp(
    min=0, max=h + 1
)
# num_classes = h + 2
index = index.view(K, 2, 2 * K)[:, 0, :]
print(index.shape)
onehot = one_hot(index, num_classes=h + 2)[..., 1:-1].transpose(1, 0)
print(onehot.reshape(2 * K, h * K).float().shape)

torch.Size([144, 288])
torch.Size([288, 1152])


In [80]:
atom_attention_encoder = AtomAttentionEncoder(
    atom_s=full_embedder_args["atom_s"],
    atom_z=full_embedder_args["atom_z"],
    token_s=full_embedder_args["token_s"],
    token_z=full_embedder_args["token_z"],
    atoms_per_window_queries=full_embedder_args["atoms_per_window_queries"],
    atoms_per_window_keys=full_embedder_args["atoms_per_window_keys"],
    atom_feature_dim=full_embedder_args["atom_feature_dim"],
    atom_encoder_depth=full_embedder_args["atom_encoder_depth"],
    atom_encoder_heads=full_embedder_args["atom_encoder_heads"],
    structure_prediction=False,
)

In [81]:
# p: B, K (N // W), H, 1, C
a, q, c, p, to_keys = atom_attention_encoder(feats)

atom_ref_pos torch.Size([1, 4608, 3])
d before torch.Size([1, 144, 32, 128, 3])
p before torch.Size([1, 144, 32, 128, 16])
c before torch.Size([1, 4608, 128])
c to p trans q torch.Size([1, 144, 32, 1, 128])
p shape torch.Size([1, 144, 32, 128, 16])
q shape torch.Size([1, 4608, 128])
atom mask torch.Size([1, 4608])
p shape torch.Size([1, 144, 32, 128, 16])
H 32 W 128 N 4608
atom_to_token torch.Size([1, 4608, 512])
q_to_a torch.Size([1, 4608, 384])
atom_to_token_mean torch.Size([1, 4608, 512])


In [35]:
from boltz.model.model import InputEmbedder
from boltz.model.modules.encoders import RelativePositionEncoder
import boltz.data.const as const

In [36]:
token_s = full_embedder_args["token_s"]
token_z = full_embedder_args["token_z"]
atom_s = full_embedder_args["atom_s"]
atom_z = full_embedder_args["atom_z"]
atoms_per_window_queries = full_embedder_args["atoms_per_window_queries"]
atoms_per_window_keys = full_embedder_args["atoms_per_window_keys"]

In [42]:
# Layer
input_embedder_layer = InputEmbedder(**full_embedder_args)
print(input_embedder_layer(feats).shape)

# token_s + 2 * const.num_tokens + 1 + len(const.pocket_contact_info)
# Input projections
s_input_dim = (token_s + 2 * const.num_tokens + 1 + len(const.pocket_contact_info))
s_init_layer = nn.Linear(s_input_dim, token_s, bias=False)
z_init_1_layer = nn.Linear(s_input_dim, token_z, bias=False)
z_init_2_layer = nn.Linear(s_input_dim, token_z, bias=False)
rel_pos_layer = RelativePositionEncoder(token_z)
token_bonds_layer = nn.Linear(1, token_z, bias=False)

# max_atom = 512 (N_atom)
print("token_s", token_s)
print("token_z", token_z)

torch.Size([1, 512, 455])
token_s 384
token_z 128


In [47]:
# Understanding MSA Module and Pairformer Module
from boltz.model.modules.trunk import MSAModule, PairformerModule
msa_module = MSAModule()

3

In [ ]:
# Autoguidance on the conditioned generation
# Initialize the sequence and pairwise embeddings

In [44]:
# Initialize the sequence and pairwise embeddings
s_inputs = input_embedder_layer(feats)
s_init = s_init_layer(s_inputs)
z_init = (
    z_init_1_layer(s_inputs)[:, :, None]
    + z_init_2_layer(s_inputs)[:, None, :]
)
relative_position_encoding = rel_pos_layer(feats)
z_init = z_init + relative_position_encoding
z_init = z_init + token_bonds_layer(feats["token_bonds"].float())

# Perform rounds of the pairwise stack
s = torch.zeros_like(s_init)
z = torch.zeros_like(z_init)

print("s and z")
print(s.shape)
print(z.shape)

# Compute pairwise mask
mask = feats["token_pad_mask"].float()
print(mask.shape)
pair_mask = mask[:, :, None] * mask[:, None, :]
print(pair_mask.shape)

recycling_steps = cfg.model.training_args.recycling_steps

s_recycle_layer = nn.Linear(token_s, token_s, bias=False)
z_recycle_layer = nn.Linear(token_z, token_z, bias=False)
s_norm_layer = nn.LayerNorm(token_s)
z_norm_layer = nn.LayerNorm(token_z)


for i in range(cfg.training_args.recycling_steps + 1):

    # Only set grad enabled for the last recycling step
    with torch.set_grad_enabled(i == recycling_steps):
        # Fixes an issue with unused parameters in autocast
        # Apply recycling (more like a skip connection with original s_init)
        s = s_init + s_recycle_layer(s_norm_layer(s))
        z = z_init + z_recycle_layer(z_norm_layer(z))

        # Compute pairwise stack
        if not self.no_msa:
            z = z + msa_module(z, s_inputs, feats)

        # Revert to uncompiled version for validation
        if self.is_pairformer_compiled and not self.training:
            pairformer_module = self.pairformer_module._orig_mod  # noqa: SLF001
        else:
            pairformer_module = self.pairformer_module

        s, z = pairformer_module(s, z, mask=mask, pair_mask=pair_mask)

pdistogram = self.distogram_module(z)
dict_out = {"pdistogram": pdistogram}

s and z
torch.Size([1, 512, 384])
torch.Size([1, 512, 512, 128])
torch.Size([1, 512])
torch.Size([1, 512, 512])


In [62]:
from boltz.model.modules.diffusion import AtomDiffusion

In [ ]:
# diffusion_process_args:
#     sigma_min: 0.0004
#     sigma_max: 160.0
#     sigma_data: 16.0
#     rho: 7
#     P_mean: -1.2
#     P_std: 1.5
#     gamma_0: 0.8
#     gamma_min: 1.0
#     noise_scale: 1.0
#     step_scale: 1.0
#     coordinate_augmentation: true
#     alignment_reverse_diff: true
#     synchronize_sigmas: true
#     use_inference_model_cache: true

In [ ]:
# Structure Module

structure_module = AtomDiffusion(
    score_model_args={
        "token_z": token_z,
        "token_s": token_s,
        "atom_z": atom_z,
        "atom_s": atom_s,
        "atoms_per_window_queries": atoms_per_window_queries,
        "atoms_per_window_keys": atoms_per_window_keys,
        "atom_feature_dim": atom_feature_dim,
        **score_model_args,
    },
    compile_score=compile_structure,
    accumulate_token_repr=use_accumulate_token_repr,
    **diffusion_process_args,
)

### Confidence Module

### Performance Benchmarking

In [ ]:
# Cropping 